In [3]:
import pandas as pd
import numpy as np
import os

# LOAD DATA
df = pd.read_csv(r"C:\Users\DELL\RetainX\data\raw\bank_churn_raw.csv")

print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

# DROP USELESS COLUMNS
drop_cols = []

for col in ["RowNumber", "Surname"]:

    if col in df.columns:
        drop_cols.append(col)

df.drop(columns=drop_cols, inplace=True)

# HANDLE MISSING VALUES

num_cols = df.select_dtypes(
    include=[np.number]
).columns

cat_cols = df.select_dtypes(
    include=["object"]
).columns

for col in num_cols:

    df[col].fillna(
        df[col].median(),
        inplace=True
    )

for col in cat_cols:

    df[col].fillna(
        df[col].mode()[0],
        inplace=True
    )

# REMOVE DUPLICATES

if "CustomerId" in df.columns:

    df.drop_duplicates(
        subset=["CustomerId"],
        inplace=True
    )

# STANDARDIZE TEXT

for col in cat_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

# OUTLIER CAPPING

def cap_outliers(df, col):

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(
        lower,
        upper
    )

    return df

for col in [
    "CreditScore",
    "Age",
    "Balance",
    "EstimatedSalary"
]:

    if col in df.columns:

        df = cap_outliers(df, col)

# SAVE CLEANED DATA

os.makedirs(
    "processed",
    exist_ok=True
)

df.to_csv(
    "processed/bank_clean.csv",
    index=False
)

print("Cleaning Complete")

  customer_id   age  gender  geography  tenure_years  num_products  \
0   CUST00001  48.0  Female    Chennai           9.0           1.0   
1   CUST00002  40.0    Male    Chennai           7.0           1.0   
2   CUST00003  50.0    Male  Hyderabad           0.0           1.0   
3   CUST00004   NaN  Female    Chennai          14.0           3.0   
4   CUST00005  39.0    Male       Pune           4.0           2.0   

   has_credit_card  is_active_member  credit_score  account_balance  ...  \
0              1.0               1.0         850.0         65450.59  ...   
1              1.0               1.0         696.0        205404.13  ...   
2              1.0               1.0         600.0         57161.98  ...   
3              1.0               0.0         678.0         27691.38  ...   
4              0.0               1.0         792.0         35135.31  ...   

   last_interaction_days_ago  mobile_app_logins_monthly  net_banking_active  \
0                      239.0               

C:\Users\DELL\AppData\Local\Temp\ipykernel_8220\1160451842.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(
C:\Users\DELL\AppData\Local\Temp\ipykernel_8220\1160451842.py:35: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inpl

Cleaning Complete


In [4]:
print(df.columns.tolist())

['customer_id', 'age', 'gender', 'geography', 'tenure_years', 'num_products', 'has_credit_card', 'is_active_member', 'credit_score', 'account_balance', 'estimated_salary', 'num_transactions_6m', 'avg_transaction_amount', 'num_complaints', 'last_interaction_days_ago', 'mobile_app_logins_monthly', 'net_banking_active', 'branch_visits_year', 'has_loan', 'loan_default_history', 'emi_to_income_ratio', 'joining_date', 'last_transaction_date', 'churn']


In [5]:

# CONVERT NUMERIC COLUMNS


numeric_cols = [
    "age",
    "tenure_years",
    "num_products",
    "has_credit_card",
    "is_active_member",
    "credit_score",
    "account_balance",
    "estimated_salary",
    "num_transactions_6m",
    "avg_transaction_amount",
    "num_complaints",
    "last_interaction_days_ago",
    "mobile_app_logins_monthly",
    "net_banking_active",
    "branch_visits_year",
    "has_loan",
    "loan_default_history",
    "emi_to_income_ratio",
    "churn"
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df.dtypes)

customer_id                      str
age                          float64
gender                           str
geography                        str
tenure_years                 float64
num_products                 float64
has_credit_card              float64
is_active_member             float64
credit_score                 float64
account_balance              float64
estimated_salary             float64
num_transactions_6m          float64
avg_transaction_amount       float64
num_complaints               float64
last_interaction_days_ago    float64
mobile_app_logins_monthly    float64
net_banking_active           float64
branch_visits_year           float64
has_loan                     float64
loan_default_history         float64
emi_to_income_ratio          float64
joining_date                     str
last_transaction_date            str
churn                          int64
dtype: object


In [6]:
print(df[
    [
        "is_active_member",
        "has_credit_card",
        "mobile_app_logins_monthly",
        "net_banking_active"
    ]
].head())

print(df[
    [
        "is_active_member",
        "has_credit_card",
        "mobile_app_logins_monthly",
        "net_banking_active"
    ]
].dtypes)

   is_active_member  has_credit_card  mobile_app_logins_monthly  \
0               1.0              1.0                        5.0   
1               1.0              1.0                       36.0   
2               1.0              1.0                       36.0   
3               0.0              1.0                       49.0   
4               1.0              0.0                       24.0   

   net_banking_active  
0                 1.0  
1                 NaN  
2                 NaN  
3                 0.0  
4                 NaN  
is_active_member             float64
has_credit_card              float64
mobile_app_logins_monthly    float64
net_banking_active           float64
dtype: object


In [7]:
cols = [
    "is_active_member",
    "has_credit_card",
    "mobile_app_logins_monthly",
    "net_banking_active",
    "num_products",
    "account_balance",
    "estimated_salary"
]

for col in cols:

    df[col] = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .astype(float)
    )

In [8]:
df["digital_engagement_score"] = (
    (
        df["is_active_member"] * 35
    )
    +
    (
        df["has_credit_card"] * 20
    )
    +
    (
        df["net_banking_active"] * 25
    )
    +
    (
        (
            df["mobile_app_logins_monthly"]
            /
            df["mobile_app_logins_monthly"].max()
        ) * 20
    )
).round(2)

In [9]:

# FEATURE ENGINEERING


import pandas as pd
import numpy as np

# LOAD CLEANED DATASET


df = pd.read_csv(
    "processed/bank_clean.csv"
)


# STANDARDIZE COLUMN NAMES

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# CONVERT NUMERIC COLUMNS


numeric_cols = [
    "age",
    "tenure_years",
    "num_products",
    "has_credit_card",
    "is_active_member",
    "credit_score",
    "account_balance",
    "estimated_salary",
    "num_transactions_6m",
    "avg_transaction_amount",
    "num_complaints",
    "last_interaction_days_ago",
    "mobile_app_logins_monthly",
    "net_banking_active",
    "branch_visits_year",
    "has_loan",
    "loan_default_history",
    "emi_to_income_ratio",
    "churn"
]

for col in numeric_cols:

    df[col] = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .astype(float)
    )


# HANDLE NULLS AFTER CONVERSION


for col in numeric_cols:

    df[col] = df[col].fillna(
        df[col].median()
    )


# 1. BALANCE TO SALARY RATIO


df["balance_salary_ratio"] = (
    df["account_balance"]
    /
    (df["estimated_salary"] + 1)
)


# 2. PRODUCTS PER TENURE


df["products_per_tenure"] = (
    df["num_products"]
    /
    (df["tenure_years"] + 1)
)

# 3. DIGITAL ENGAGEMENT SCORE


df["digital_engagement_score"] = (
    (
        df["is_active_member"] * 35
    )
    +
    (
        df["has_credit_card"] * 20
    )
    +
    (
        df["net_banking_active"] * 25
    )
    +
    (
        (
            df["mobile_app_logins_monthly"]
            /
            df["mobile_app_logins_monthly"].max()
        ) * 20
    )
).round(2)


# 4. CUSTOMER VALUE SCORE


df["customer_value"] = (
    (
        df["account_balance"]
        /
        df["account_balance"].max()
    ) * 0.35
    +
    (
        df["estimated_salary"]
        /
        df["estimated_salary"].max()
    ) * 0.25
    +
    (
        df["tenure_years"]
        /
        df["tenure_years"].max()
    ) * 0.15
    +
    (
        df["num_products"]
        /
        df["num_products"].max()
    ) * 0.15
    +
    (
        df["avg_transaction_amount"]
        /
        df["avg_transaction_amount"].max()
    ) * 0.10
).round(4)

# 5. REVENUE AT RISK


df["revenue_at_risk"] = (
    (
        df["account_balance"] * 0.02
    )
    +
    (
        df["estimated_salary"] * 0.004
    )
    +
    (
        df["num_products"] * 400
    )
    +
    (
        df["avg_transaction_amount"] * 0.01
    )
).round(2)

# 6. AGE TENURE RATIO


df["age_tenure_ratio"] = (
    df["age"]
    /
    (df["tenure_years"] + 1)
)

# 7. ZERO BALANCE FLAG


df["zero_balance_flag"] = (
    df["account_balance"] == 0
).astype(int)


# 8. SINGLE PRODUCT FLAG


df["single_product_flag"] = (
    df["num_products"] == 1
).astype(int)


# 9. CREDIT ENGAGEMENT MISMATCH


df["credit_engagement_mismatch"] = (
    (
        df["credit_score"] > 700
    )
    &
    (
        df["is_active_member"] == 0
    )
).astype(int)


# 10. COMPLAINT RISK SCORE


df["complaint_risk_score"] = (
    (
        df["num_complaints"]
        *
        df["last_interaction_days_ago"]
    )
    / 10
).round(2)


# 11. LOW DIGITAL ACTIVITY FLAG


df["low_digital_activity"] = (
    (
        df["mobile_app_logins_monthly"] < 5
    )
    &
    (
        df["net_banking_active"] == 0
    )
).astype(int)


# 12. FINANCIAL STRESS FLAG


df["financial_stress_flag"] = (
    df["emi_to_income_ratio"] > 0.5
).astype(int)


# 13. WEALTH TIER


df["wealth_tier"] = pd.cut(
    df["estimated_salary"],
    bins=[0, 50000, 100000, 150000, 300000],
    labels=[
        "Low",
        "Mid",
        "High",
        "Premium"
    ]
)


# 14. AGE GROUP


df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 30, 45, 60, 100],
    labels=[
        "Young",
        "Mid-Career",
        "Senior",
        "Retired"
    ]
)


# 15. CHURN RISK SCORE


df["churn_risk_score"] = (
    (
        1 - df["is_active_member"]
    ) * 25
    +
    df["zero_balance_flag"] * 20
    +
    df["single_product_flag"] * 15
    +
    (
        df["age"] > 55
    ).astype(int) * 10
    +
    (
        df["tenure_years"] < 3
    ).astype(int) * 10
    +
    df["credit_engagement_mismatch"] * 10
    +
    df["financial_stress_flag"] * 15
    +
    df["low_digital_activity"] * 10
    +
    (
        df["num_complaints"] > 2
    ).astype(int) * 10
)


# 16. PRIORITY SCORE


df["priority_score"] = (
    (
        df["customer_value"]
        *
        df["revenue_at_risk"]
    )
    /
    (
        df["churn_risk_score"] + 1
    )
).round(2)


# 17. HIGH VALUE CUSTOMER FLAG


df["high_value_customer"] = (
    df["customer_value"] > 0.6
).astype(int)


# 18. HIGH CHURN RISK FLAG


df["high_churn_risk"] = (
    df["churn_risk_score"] > 50
).astype(int)


# FINAL OUTPUT


print("\nFeature Engineering Complete")

print("\nNew Shape:")
print(df.shape)

print("\nNew Features Added Successfully")

print(df.head())


# SAVE FEATURE ENGINEERED DATASET


df.to_csv(
    "processed/bank_churn_features.csv",
    index=False
)

print(
    "\nFeature engineered dataset saved successfully"
)


Feature Engineering Complete

New Shape:
(10100, 42)

New Features Added Successfully
  customer_id   age  gender  geography  tenure_years  num_products  \
0   Cust00001  48.0  Female    Chennai           9.0           1.0   
1   Cust00002  40.0    Male    Chennai           7.0           1.0   
2   Cust00003  50.0    Male  Hyderabad           0.0           1.0   
3   Cust00004  42.0  Female    Chennai          14.0           3.0   
4   Cust00005  39.0    Male       Pune           4.0           2.0   

   has_credit_card  is_active_member  credit_score  account_balance  ...  \
0              1.0               1.0         850.0         65450.59  ...   
1              1.0               1.0         696.0        205404.13  ...   
2              1.0               1.0         600.0         57161.98  ...   
3              1.0               0.0         678.0         27691.38  ...   
4              0.0               1.0         792.0         35135.31  ...   

   credit_engagement_mismatch  comp

In [10]:

# BUSINESS EDA


import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os


# LOAD FEATURE ENGINEERED DATA


df = pd.read_csv(
    "processed/bank_churn_features.csv"
)

# CREATE FOLDER


os.makedirs(
    "eda_plots",
    exist_ok=True
)


# BASIC CHURN RATE


print("\n===== CHURN RATE =====")

print(
    df["churn"]
    .value_counts(normalize=True)
    .round(3)
)


# Q1 — WHO CHURNS? (AGE ANALYSIS)


fig = px.histogram(
    df,
    x="age",
    color="churn",
    barmode="overlay",
    title="Age Distribution by Churn",
    color_discrete_map={
        0: "#2ecc71",
        1: "#e74c3c"
    }
)

fig.write_html(
    "eda_plots/age_churn.html"
)


# Q2 — WHICH GEOGRAPHY HAS HIGHEST CHURN?


geo_churn = (
    df.groupby("geography")["churn"]
    .mean()
    .reset_index()
)

geo_churn.columns = [
    "Geography",
    "Churn Rate"
]

fig2 = px.bar(
    geo_churn,
    x="Geography",
    y="Churn Rate",
    title="Churn Rate by Geography",
    color="Churn Rate",
    color_continuous_scale="Reds"
)

fig2.write_html(
    "eda_plots/geography_churn.html"
)

# Q3 — FINANCIAL IMPACT


total_revenue_at_risk = (
    df[
        df["churn"] == 1
    ]["revenue_at_risk"].sum()
)

print(
    f"\nTotal Revenue At Risk: ${total_revenue_at_risk:,.0f}"
)

# Q4 — ACTIVE VS INACTIVE CHURN


active_churn = (
    df.groupby("is_active_member")["churn"]
    .mean()
)

print(
    "\nChurn by Active Status:\n"
)

print(active_churn)

# Q5 — DIGITAL ENGAGEMENT VS CHURN


fig3 = px.box(
    df,
    x="churn",
    y="digital_engagement_score",
    color="churn",
    title="Digital Engagement vs Churn"
)

fig3.write_html(
    "eda_plots/digital_engagement_churn.html"
)


# Q6 — COMPLAINTS VS CHURN


complaint_churn = (
    df.groupby("num_complaints")["churn"]
    .mean()
    .reset_index()
)

fig4 = px.line(
    complaint_churn,
    x="num_complaints",
    y="churn",
    markers=True,
    title="Complaints vs Churn Rate"
)

fig4.write_html(
    "eda_plots/complaints_churn.html"
)

# Q7 — CUSTOMER VALUE VS CHURN


fig5 = px.scatter(
    df,
    x="customer_value",
    y="revenue_at_risk",
    color="churn",
    size="priority_score",
    title="Customer Value vs Revenue Risk"
)

fig5.write_html(
    "eda_plots/customer_value_risk.html"
)

# Q8 — CORRELATION ANALYSIS


num_df = df.select_dtypes(
    include=[np.number]
)

corr = (
    num_df.corr()["churn"]
    .drop("churn")
    .sort_values()
)

print(
    "\nTop Factors Correlated with Churn:\n"
)

print(corr.tail(10))


# Q9 — WEALTH TIER CHURN


wealth_churn = (
    df.groupby("wealth_tier")["churn"]
    .mean()
    .reset_index()
)

fig6 = px.bar(
    wealth_churn,
    x="wealth_tier",
    y="churn",
    color="churn",
    title="Churn by Wealth Tier"
)

fig6.write_html(
    "eda_plots/wealth_tier_churn.html"
)


# Q10 — AGE GROUP CHURN


age_group_churn = (
    df.groupby("age_group")["churn"]
    .mean()
    .reset_index()
)

fig7 = px.bar(
    age_group_churn,
    x="age_group",
    y="churn",
    color="churn",
    title="Churn by Age Group"
)

fig7.write_html(
    "eda_plots/age_group_churn.html"
)

# Q11 — HIGH RISK CUSTOMER ANALYSIS


high_risk_customers = df[
    df["high_churn_risk"] == 1
]

print(
    "\n===== HIGH RISK CUSTOMERS ====="
)

print(
    "Count:",
    len(high_risk_customers)
)

print(
    "Average Revenue Risk:",
    round(
        high_risk_customers[
            "revenue_at_risk"
        ].mean(),
        2
    )
)


# Q12 — EXECUTIVE SUMMARY


print("\n===== EXECUTIVE INSIGHTS =====")

print(
    f"""
1. Total Revenue At Risk:
   ${total_revenue_at_risk:,.0f}

2. Highest Churn Geography:
   {geo_churn.sort_values(
       by='Churn Rate',
       ascending=False
   ).iloc[0]['Geography']}

3. Most Important Churn Factors:
   {corr.tail(3).index.tolist()}

4. High Risk Customers:
   {len(high_risk_customers)}

5. Avg Revenue Risk per High-Risk Customer:
   ${round(high_risk_customers['revenue_at_risk'].mean(),2)}
"""
)

# FINAL MESSAGE


print(
    "\nEDA Complete"
)

print(
    "\nPlots saved in: eda_plots/"
)


===== CHURN RATE =====
churn
0.0    0.784
1.0    0.216
Name: proportion, dtype: float64

Total Revenue At Risk: $7,996,529

Churn by Active Status:

is_active_member
0.0    0.319245
1.0    0.160586
Name: churn, dtype: float64

Top Factors Correlated with Churn:

customer_value                0.043880
age_tenure_ratio              0.046928
loan_default_history          0.049030
products_per_tenure           0.052198
num_products                  0.067198
high_churn_risk               0.070543
credit_engagement_mismatch    0.082900
churn_risk_score              0.168703
zero_balance_flag                  NaN
high_value_customer                NaN
Name: churn, dtype: float64

===== HIGH RISK CUSTOMERS =====
Count: 454
Average Revenue Risk: 3802.07

===== EXECUTIVE INSIGHTS =====

1. Total Revenue At Risk:
   $7,996,529

2. Highest Churn Geography:
   Hyderabad

3. Most Important Churn Factors:
   ['churn_risk_score', 'zero_balance_flag', 'high_value_customer']

4. High Risk Customers:
  

In [12]:

# PREPROCESSING

import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.model_selection import (
    train_test_split
)

from imblearn.over_sampling import (
    SMOTE
)

import joblib
import os

# LOAD FEATURE ENGINEERED DATASET


df = pd.read_csv(
    "processed/bank_churn_features.csv"
)


# STANDARDIZE COLUMN NAMES


df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# DROP NON-ML COLUMNS


drop_cols = [
    "customer_id",
    "joining_date",
    "last_transaction_date"
]

df_ml = df.drop(
    columns=drop_cols,
    errors="ignore"
)

# ENCODE CATEGORICAL COLUMNS


categorical_cols = [
    "gender",
    "geography",
    "wealth_tier",
    "age_group"
]

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    df_ml[col] = le.fit_transform(
        df_ml[col].astype(str)
    )

    encoders[col] = le


# FEATURES + TARGET

TARGET = "churn"

X = df_ml.drop(
    columns=[TARGET]
)

y = df_ml[TARGET]


# CLASS DISTRIBUTION


print(
    "\nClass Distribution Before SMOTE:"
)

print(
    y.value_counts().to_dict()
)


# TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

# HANDLE CLASS IMBALANCE USING SMOTE


smote = SMOTE(
    random_state=42
)

X_train_res, y_train_res = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

print(
    "\nClass Distribution After SMOTE:"
)

print(
    pd.Series(
        y_train_res
    ).value_counts().to_dict()
)


# FEATURE SCALING


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_res
)

X_test_scaled = scaler.transform(
    X_test
)


# CREATE MODELS FOLDER


os.makedirs(
    "models",
    exist_ok=True
)

os.makedirs(
    "processed",
    exist_ok=True
)


# SAVE SCALER


joblib.dump(
    scaler,
    "models/scaler.joblib"
)


# SAVE ENCODERS


for col, encoder in encoders.items():

    joblib.dump(
        encoder,
        f"models/{col}_encoder.joblib"
    )


# SAVE FEATURE NAMES


joblib.dump(
    X.columns.tolist(),
    "models/feature_names.joblib"
)


# SAVE TEST SETS

X_test.to_csv(
    "processed/X_test.csv",
    index=False
)

y_test.to_csv(
    "processed/y_test.csv",
    index=False
)

# SAVE NUMPY ARRAYS


np.save(
    "processed/X_train_scaled.npy",
    X_train_scaled
)

np.save(
    "processed/X_test_scaled.npy",
    X_test_scaled
)

np.save(
    "processed/y_train_res.npy",
    y_train_res
)

np.save(
    "processed/y_test.npy",
    y_test.values
)


# FINAL OUTPUT


print(
    "\nPreprocessing Complete"
)

print(
    "\nFiles Saved Successfully:"
)

print(
    """
1. scaler.joblib
2. encoders
3. feature_names.joblib
4. X_train_scaled.npy
5. X_test_scaled.npy
6. y_train_res.npy
7. y_test.npy
"""
)

print(
    "\nFinal Training Shape:"
)

print(
    X_train_scaled.shape
)

print(
    "\nFinal Testing Shape:"
)

print(
    X_test_scaled.shape
)


Class Distribution Before SMOTE:
{0.0: 7915, 1.0: 2185}

Class Distribution After SMOTE:
{0.0: 6332, 1.0: 6332}

Preprocessing Complete

Files Saved Successfully:

1. scaler.joblib
2. encoders
3. feature_names.joblib
4. X_train_scaled.npy
5. X_test_scaled.npy
6. y_train_res.npy
7. y_test.npy


Final Training Shape:
(12664, 38)

Final Testing Shape:
(2020, 38)


In [13]:
#  MODEL TRAINING


import numpy as np
import pandas as pd
import joblib
import warnings

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.ensemble import (
    RandomForestClassifier
)

from xgboost import (
    XGBClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# LOAD PREPROCESSED DATA


X_train = np.load(
    "processed/X_train_scaled.npy"
)

X_test = np.load(
    "processed/X_test_scaled.npy"
)

y_train = np.load(
    "processed/y_train_res.npy"
)

y_test = np.load(
    "processed/y_test.npy"
)


# EVALUATION FUNCTION


def evaluate(
    name,
    model,
    X_test,
    y_test
):

    y_pred = model.predict(
        X_test
    )

    y_prob = model.predict_proba(
        X_test
    )[:,1]

    metrics = {

        "Model":
            name,

        "Accuracy":
            round(
                accuracy_score(
                    y_test,
                    y_pred
                ),
                4
            ),

        "Precision":
            round(
                precision_score(
                    y_test,
                    y_pred
                ),
                4
            ),

        "Recall":
            round(
                recall_score(
                    y_test,
                    y_pred
                ),
                4
            ),

        "F1 Score":
            round(
                f1_score(
                    y_test,
                    y_pred
                ),
                4
            ),

        "ROC-AUC":
            round(
                roc_auc_score(
                    y_test,
                    y_prob
                ),
                4
            )
    }

    print("\n")
    print("=" * 60)

    print(f"{name}")

    print("=" * 60)

    for k, v in metrics.items():

        if k != "Model":

            print(
                f"{k:15}: {v}"
            )

    print("\nClassification Report:\n")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "Retained",
                "Churned"
            ]
        )
    )

    print("\nConfusion Matrix:\n")

    print(
        confusion_matrix(
            y_test,
            y_pred
        )
    )

    return metrics


# MODEL 1 — LOGISTIC REGRESSION


print(
    "\nTraining Logistic Regression..."
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=0.1
)

lr.fit(
    X_train,
    y_train
)

m1 = evaluate(
    "Logistic Regression",
    lr,
    X_test,
    y_test
)

joblib.dump(
    lr,
    "models/logistic_regression.joblib"
)


# MODEL 2 — RANDOM FOREST


print(
    "\nTraining Random Forest..."
)

rf = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    min_samples_split=5,

    min_samples_leaf=2,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

m2 = evaluate(
    "Random Forest",
    rf,
    X_test,
    y_test
)

joblib.dump(
    rf,
    "models/random_forest.joblib"
)


# MODEL 3 — XGBOOST


print(
    "\nTraining XGBoost..."
)

xgb = XGBClassifier(

    n_estimators=400,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=3,

    gamma=1,

    reg_alpha=0.5,

    reg_lambda=1,

    eval_metric="auc",

    random_state=42,

    n_jobs=-1
)

xgb.fit(

    X_train,

    y_train,

    eval_set=[
        (
            X_test,
            y_test
        )
    ],

    verbose=50
)

m3 = evaluate(
    "XGBoost",
    xgb,
    X_test,
    y_test
)

joblib.dump(
    xgb,
    "models/xgboost_final.joblib"
)


# MODEL COMPARISON TABLE


results = pd.DataFrame(
    [m1, m2, m3]
)

results.set_index(
    "Model",
    inplace=True
)

print("\n")
print("=" * 70)

print(
    "MODEL COMPARISON TABLE"
)

print("=" * 70)

print(
    results.to_string()
)

# SAVE RESULTS

results.to_csv(
    "processed/model_results.csv"
)


# FEATURE IMPORTANCE


feature_names = joblib.load(
    "models/feature_names.joblib"
)

importance_df = pd.DataFrame({

    "Feature":
        feature_names,

    "Importance":
        xgb.feature_importances_

})

importance_df = (
    importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
)

importance_df.to_csv(
    "processed/feature_importance.csv",
    index=False
)

print("\nTop 10 Important Features:\n")

print(
    importance_df.head(10)
)


# FINAL BUSINESS INSIGHTS


best_model_auc = results[
    "ROC-AUC"
].max()

print("\n")
print("=" * 70)

print(
    "BUSINESS INSIGHTS"
)

print("=" * 70)

print(f"""

1. Best Model:
   XGBoost

2. Best ROC-AUC:
   {best_model_auc}

3. Model Purpose:
   Predict customer churn risk

4. Business Value:
   Helps banks identify high-risk customers,
   prioritize retention campaigns,
   and reduce revenue loss.

5. Strategic Capability:
   Enables AI-powered customer retention decisions.

""")


# FINAL MESSAGE


print(
    "\nAll models trained successfully"
)

print(
    "\nSaved Models:"
)

print("""
1. logistic_regression.joblib
2. random_forest.joblib
3. xgboost_final.joblib
""")

print(
    "\nBest Production Model: XGBoost"
)


Training Logistic Regression...


Logistic Regression
Accuracy       : 0.6812
Precision      : 0.3122
Recall         : 0.3936
F1 Score       : 0.3482
ROC-AUC        : 0.6104

Classification Report:

              precision    recall  f1-score   support

    Retained       0.82      0.76      0.79      1583
     Churned       0.31      0.39      0.35       437

    accuracy                           0.68      2020
   macro avg       0.57      0.58      0.57      2020
weighted avg       0.71      0.68      0.69      2020


Confusion Matrix:

[[1204  379]
 [ 265  172]]

Training Random Forest...


Random Forest
Accuracy       : 0.7772
Precision      : 0.4156
Recall         : 0.0732
F1 Score       : 0.1245
ROC-AUC        : 0.6334

Classification Report:

              precision    recall  f1-score   support

    Retained       0.79      0.97      0.87      1583
     Churned       0.42      0.07      0.12       437

    accuracy                           0.78      2020
   macro avg       0

In [14]:
# SHAP EXPLAINABILITY


import numpy as np
import pandas as pd
import shap
import joblib
import matplotlib.pyplot as plt
import os


# LOAD MODEL AND DATA


xgb = joblib.load(
    "models/xgboost_final.joblib"
)

X_test = pd.read_csv(
    "processed/X_test.csv"
)

feature_names = joblib.load(
    "models/feature_names.joblib"
)

# ENSURE COLUMN NAMES MATCH

X_test.columns = feature_names


# CREATE OUTPUT FOLDERS


os.makedirs(
    "shap_outputs",
    exist_ok=True
)

os.makedirs(
    "processed",
    exist_ok=True
)


# COMPUTE SHAP VALUES


print(
    "\nComputing SHAP values..."
)

explainer = shap.TreeExplainer(
    xgb
)

shap_values = explainer.shap_values(
    X_test
)

print(
    "\nSHAP values computed successfully"
)


# 1. GLOBAL FEATURE IMPORTANCE


plt.figure(figsize=(12,8))

shap.summary_plot(

    shap_values,

    X_test,

    plot_type="bar",

    show=False
)

plt.title(
    "Global Feature Importance"
)

plt.tight_layout()

plt.savefig(

    "shap_outputs/shap_feature_importance.png",

    dpi=200,

    bbox_inches="tight"
)

plt.close()

print(
    "\nGlobal feature importance plot saved"
)


# 2. SHAP BEESWARM PLOT


plt.figure(figsize=(12,8))

shap.summary_plot(

    shap_values,

    X_test,

    show=False
)

plt.title(
    "SHAP Beeswarm Plot"
)

plt.tight_layout()

plt.savefig(

    "shap_outputs/shap_beeswarm.png",

    dpi=200,

    bbox_inches="tight"
)

plt.close()

print(
    "\nSHAP beeswarm plot saved"
)


# 3. WATERFALL PLOT FOR HIGH-RISK CUSTOMER


y_test = np.load(
    "processed/y_test.npy"
)

# FIRST ACTUAL CHURNER

churner_idx = np.where(
    y_test == 1
)[0][0]

plt.figure(figsize=(12,8))

shap.waterfall_plot(

    shap.Explanation(

        values=shap_values[churner_idx],

        base_values=explainer.expected_value,

        data=X_test.iloc[churner_idx],

        feature_names=feature_names
    ),

    show=False
)

plt.tight_layout()

plt.savefig(

    "shap_outputs/shap_individual_waterfall.png",

    dpi=200,

    bbox_inches="tight"
)

plt.close()

print(
    "\nIndividual customer explanation saved"
)


# 4. DEPENDENCE PLOT


top_feature = feature_names[
    np.argmax(
        np.abs(shap_values).mean(0)
    )
]

plt.figure(figsize=(10,6))

shap.dependence_plot(

    top_feature,

    shap_values,

    X_test,

    show=False
)

plt.tight_layout()

plt.savefig(

    "shap_outputs/shap_dependence_plot.png",

    dpi=200,

    bbox_inches="tight"
)

plt.close()

print(
    "\nDependence plot saved"
)


# 5. SAVE SHAP VALUES


shap_df = pd.DataFrame(

    shap_values,

    columns=feature_names
)

shap_df.to_csv(

    "processed/shap_values.csv",

    index=False
)

print(
    "\nSHAP values exported successfully"
)


# 6. TOP FEATURES TABLE


importance_df = pd.DataFrame({

    "Feature":
        feature_names,

    "Mean_SHAP_Importance":
        np.abs(shap_values).mean(axis=0)

})

importance_df = (

    importance_df

    .sort_values(

        by="Mean_SHAP_Importance",

        ascending=False
    )
)

importance_df.to_csv(

    "processed/shap_feature_importance.csv",

    index=False
)

print(
    "\nTop SHAP Features:\n"
)

print(
    importance_df.head(10)
)

# BUSINESS INTERPRETATION


print("\n")
print("=" * 70)

print(
    "BUSINESS INTERPRETATION"
)

print("=" * 70)

print(f"""

Top churn-driving features identified by SHAP:

{importance_df.head(5)['Feature'].tolist()}

This means the AI model mainly considers:
- customer engagement
- financial stress
- complaints
- inactivity
- customer value

while predicting churn risk.

These insights help banks:
1. identify WHY customers churn
2. build targeted retention strategies
3. prioritize high-value customers
4. reduce revenue loss

""")


# FINAL MESSAGE


print("\n")
print("=" * 70)

print(
    "SHAP EXPLAINABILITY COMPLETE"
)

print("=" * 70)

print("""

Generated Outputs:

1. shap_feature_importance.png
2. shap_beeswarm.png
3. shap_individual_waterfall.png
4. shap_dependence_plot.png
5. shap_values.csv
6. shap_feature_importance.csv

Folder:
shap_outputs/

""")


Computing SHAP values...

SHAP values computed successfully

Global feature importance plot saved

SHAP beeswarm plot saved

Individual customer explanation saved

Dependence plot saved

SHAP values exported successfully

Top SHAP Features:

                     Feature  Mean_SHAP_Importance
17                  has_loan              1.319018
18      loan_default_history              1.244339
6           is_active_member              0.799944
4               num_products              0.749517
29      complaint_risk_score              0.392981
22  digital_engagement_score              0.359756
5            has_credit_card              0.335585
10       num_transactions_6m              0.328213
12            num_complaints              0.297842
27       single_product_flag              0.282617


BUSINESS INTERPRETATION


Top churn-driving features identified by SHAP:

['has_loan', 'loan_default_history', 'is_active_member', 'num_products', 'complaint_risk_score']

This means the AI mode

<Figure size 1000x600 with 0 Axes>

In [15]:
# CUSTOMER PRIORITIZATION ENGINE


import pandas as pd
import numpy as np
import joblib


# LOAD MODEL + SCALER + FEATURES


xgb = joblib.load(
    "models/xgboost_final.joblib"
)

scaler = joblib.load(
    "models/scaler.joblib"
)

feature_names = joblib.load(
    "models/feature_names.joblib"
)


# LOAD FULL FEATURE ENGINEERED DATASET


df_full = pd.read_csv(
    "processed/bank_churn_features.csv"
)

df_ml = df_full.copy()


# STANDARDIZE COLUMN NAMES


df_ml.columns = (
    df_ml.columns
    .str.strip()
    .str.lower()
)

df_full.columns = (
    df_full.columns
    .str.strip()
    .str.lower()
)


# LOAD ENCODERS


gender_encoder = joblib.load(
    "models/gender_encoder.joblib"
)

geography_encoder = joblib.load(
    "models/geography_encoder.joblib"
)

wealth_encoder = joblib.load(
    "models/wealth_tier_encoder.joblib"
)

age_group_encoder = joblib.load(
    "models/age_group_encoder.joblib"
)


# ENCODE CATEGORICAL COLUMNS


df_ml["gender"] = gender_encoder.transform(
    df_ml["gender"].astype(str)
)

df_ml["geography"] = geography_encoder.transform(
    df_ml["geography"].astype(str)
)

df_ml["wealth_tier"] = wealth_encoder.transform(
    df_ml["wealth_tier"].astype(str)
)

df_ml["age_group"] = age_group_encoder.transform(
    df_ml["age_group"].astype(str)
)


# DROP NON-ML COLUMNS


drop_cols = [
    "customer_id",
    "joining_date",
    "last_transaction_date",
    "churn"
]

X_all = df_ml.drop(
    columns=drop_cols,
    errors="ignore"
)

# ENSURE FEATURE ORDER MATCHES TRAINING

X_all = X_all[
    feature_names
]


# SCALE FEATURES


X_all_scaled = scaler.transform(
    X_all
)

# PREDICT CHURN PROBABILITY


churn_proba = xgb.predict_proba(
    X_all_scaled
)[:,1]

df_full["churn_probability"] = (
    churn_proba.round(4)
)

# PRIORITY SCORE


df_full["priority_score"] = (

    df_full["churn_probability"]

    *

    df_full["customer_value"]

    *

    (
        df_full["revenue_at_risk"]
        /
        df_full["revenue_at_risk"].max()
    )

).round(6)


# RECOVERABLE REVENUE


RECOVERY_RATE = 0.35

df_full["recoverable_revenue"] = (

    df_full["revenue_at_risk"]

    *

    df_full["churn_probability"]

    *

    RECOVERY_RATE

).round(2)

# CUSTOMER LIFETIME VALUE


df_full["customer_lifetime_value"] = (

    (
        df_full["account_balance"] * 0.03
    )

    +

    (
        df_full["avg_transaction_amount"] * 12
    )

    +

    (
        df_full["num_products"] * 1000
    )

).round(2)


# RISK TIERS


df_full["risk_tier"] = pd.cut(

    df_full["churn_probability"],

    bins=[0, 0.3, 0.6, 0.8, 1.0],

    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Critical"
    ]
)


# RETENTION RECOMMENDATION ENGINE


def recommend_action(row):

    if (
        row["risk_tier"] == "Critical"
        and
        row["customer_value"] > 0.6
    ):

        return (
            "Assign Dedicated Relationship Manager"
        )

    elif row["single_product_flag"] == 1:

        return (
            "Cross-Sell Premium Banking Products"
        )

    elif row["low_digital_activity"] == 1:

        return (
            "Launch Digital Engagement Campaign"
        )

    elif row["financial_stress_flag"] == 1:

        return (
            "Offer Flexible EMI / Loan Support"
        )

    elif row["num_complaints"] > 2:

        return (
            "Priority Complaint Resolution"
        )

    else:

        return (
            "Standard Retention Campaign"
        )

df_full["recommended_action"] = (

    df_full.apply(
        recommend_action,
        axis=1
    )
)

# EXECUTIVE METRICS


total_revenue_risk = (
    df_full["revenue_at_risk"].sum()
)

recoverable_revenue = (
    df_full["recoverable_revenue"].sum()
)

critical_customers = (
    (
        df_full["risk_tier"]
        ==
        "Critical"
    ).sum()
)

high_risk_customers = (
    (
        df_full["risk_tier"]
        ==
        "High Risk"
    ).sum()
)


# PRINT EXECUTIVE SUMMARY


print("\n")
print("=" * 70)

print(
    "CUSTOMER RISK ANALYSIS"
)

print("=" * 70)

print(
    "\nRISK DISTRIBUTION:\n"
)

print(
    df_full["risk_tier"]
    .value_counts()
)

print("\n")
print("=" * 70)

print(
    "FINANCIAL IMPACT"
)

print("=" * 70)

print(
    f"""

Total Revenue At Risk:
${total_revenue_risk:,.0f}

Recoverable Revenue:
${recoverable_revenue:,.0f}

Critical Customers:
{critical_customers}

High Risk Customers:
{high_risk_customers}

"""
)


# TOP 10 CUSTOMERS TO SAVE


top10 = (

    df_full

    .nlargest(
        10,
        "priority_score"
    )

)[[
    "customer_id",
    "churn_probability",
    "customer_value",
    "revenue_at_risk",
    "priority_score",
    "risk_tier",
    "recommended_action"
]]

print("\n")
print("=" * 70)

print(
    "TOP 10 CUSTOMERS TO SAVE"
)

print("=" * 70)

print(
    top10.to_string(index=False)
)


# SAVE OUTPUTS


df_full.to_csv(

    "processed/customer_scores.csv",

    index=False
)

top10.to_csv(

    "processed/top10_priority_customers.csv",

    index=False
)


# FINAL MESSAGE


print("\n")
print("=" * 70)

print(
    "CUSTOMER PRIORITIZATION COMPLETE"
)

print("=" * 70)

print("""

Generated Files:

1. customer_scores.csv
2. top10_priority_customers.csv

Key Features:
- churn probability
- priority score
- revenue risk
- recoverable revenue
- risk tier
- recommended retention actions

Business Value:
Helps banks prioritize customers,
reduce churn,
and optimize retention strategy.

""")



CUSTOMER RISK ANALYSIS

RISK DISTRIBUTION:

risk_tier
Low Risk       6034
Medium Risk    2177
High Risk      1136
Critical        753
Name: count, dtype: int64


FINANCIAL IMPACT


Total Revenue At Risk:
$36,617,107

Recoverable Revenue:
$4,196,718

Critical Customers:
753

High Risk Customers:
1136




TOP 10 CUSTOMERS TO SAVE
customer_id  churn_probability  customer_value  revenue_at_risk  priority_score   risk_tier                recommended_action
  Cust01089             0.9214          0.4103        221541.21        0.348475    Critical     Priority Complaint Resolution
  Cust05371             0.7555          0.4427        240343.52        0.334460   High Risk       Standard Retention Campaign
  Cust03345             0.8184          0.4184        220264.18        0.313811    Critical     Priority Complaint Resolution
  Cust08124             0.8608          0.4194        175864.78        0.264166    Critical Offer Flexible EMI / Loan Support
  Cust01133             0.8132        

In [ ]:
# RETENTION RECOMMENDATION ENGINE


import pandas as pd


# LOAD CUSTOMER SCORES


df = pd.read_csv(
    "processed/customer_scores.csv"
)


# REMOVE OLD RECOMMENDATION COLUMNS


old_cols = [
    "recommended_action",
    "recommendation_reason",
    "expected_impact"
]

df.drop(
    columns=old_cols,
    errors="ignore",
    inplace=True
)


# RETENTION RECOMMENDATION FUNCTION


def get_retention_recommendation(row):

    """
    AI-driven recommendation engine
    """

    actions = []

    
    # RULE 1 — HIGH VALUE + HIGH CHURN
    

    if (
        row["customer_value"] > 0.6
        and
        row["churn_probability"] > 0.5
    ):

        actions.append({

            "action":
                "Assign Dedicated Relationship Manager",

            "reason":
                "High-value customer at critical churn risk",

            "expected_impact":
                "High",

            "priority":
                1
        })

    
    # RULE 2 — ZERO BALANCE CUSTOMER
    

    if row["zero_balance_flag"] == 1:

        actions.append({

            "action":
                "Offer Fixed Deposit / Savings Bonus",

            "reason":
                "Zero balance customers have low retention stickiness",

            "expected_impact":
                "Medium",

            "priority":
                2
        })

    
    # RULE 3 — SINGLE PRODUCT CUSTOMER
    

    if row["single_product_flag"] == 1:

        actions.append({

            "action":
                "Cross-Sell Credit Card or Investment Product",

            "reason":
                "Single-product customers churn significantly more",

            "expected_impact":
                "Medium",

            "priority":
                2
        })

    
    # RULE 4 — LOW DIGITAL ENGAGEMENT
    

    if row["digital_engagement_score"] < 40:

        actions.append({

            "action":
                "Digital Re-engagement Campaign",

            "reason":
                "Low digital activity indicates disengagement",

            "expected_impact":
                "Medium",

            "priority":
                3
        })

    
    # RULE 5 — INACTIVE CUSTOMER
    

    if row["is_active_member"] == 0:

        actions.append({

            "action":
                "Loyalty Rewards / Cashback Activation Offer",

            "reason":
                "Inactive customers are more likely to churn",

            "expected_impact":
                "High",

            "priority":
                1
        })

    
    # RULE 6 — PREMIUM CUSTOMER
    

    if (
        row["wealth_tier"] == "Premium"
        and
        row["churn_probability"] > 0.4
    ):

        actions.append({

            "action":
                "Priority Banking / Wealth Management Upgrade",

            "reason":
                "Premium customer not receiving premium engagement",

            "expected_impact":
                "High",

            "priority":
                1
        })

    
    # RULE 7 — HIGH COMPLAINT CUSTOMER
    

    if row["num_complaints"] > 2:

        actions.append({

            "action":
                "Priority Complaint Resolution + Follow-up",

            "reason":
                "Repeated complaints strongly linked to churn",

            "expected_impact":
                "High",

            "priority":
                1
        })

    
    # RULE 8 — FINANCIAL STRESS CUSTOMER
    

    if row["financial_stress_flag"] == 1:

        actions.append({

            "action":
                "Flexible EMI / Loan Restructuring Offer",

            "reason":
                "Customer shows financial stress indicators",

            "expected_impact":
                "Medium",

            "priority":
                2
        })

    
    # RULE 9 — LOW DIGITAL + HIGH VALUE
    

    if (
        row["low_digital_activity"] == 1
        and
        row["customer_value"] > 0.5
    ):

        actions.append({

            "action":
                "Personalized Premium Engagement Campaign",

            "reason":
                "High-value customer becoming digitally inactive",

            "expected_impact":
                "High",

            "priority":
                2
        })

    
    # DEFAULT RULE
    

    if not actions:

        actions.append({

            "action":
                "Standard Retention Email Campaign",

            "reason":
                "Low-risk customer requiring periodic engagement",

            "expected_impact":
                "Low",

            "priority":
                4
        })

    
    # SORT ACTIONS BY PRIORITY
    

    actions_sorted = sorted(
        actions,
        key=lambda x: x["priority"]
    )

    top_action = actions_sorted[0]

    
    # RETURN RESULT
    

    return pd.Series({

        "recommended_action":
            top_action["action"],

        "recommendation_reason":
            top_action["reason"],

        "expected_impact":
            top_action["expected_impact"]
    })


# APPLY RECOMMENDATION ENGINE


recommendations = df.apply(
    get_retention_recommendation,
    axis=1
)

# MERGE RESULTS


df = pd.concat(
    [df, recommendations],
    axis=1
)

# EXECUTIVE INSIGHTS


print("\n")
print("=" * 70)

print(
    "RETENTION RECOMMENDATION ANALYSIS"
)

print("=" * 70)

print("\nTop Recommended Actions:\n")

print(
    df["recommended_action"]
    .value_counts()
)


# HIGH IMPACT CUSTOMERS


high_impact = df[
    df["expected_impact"] == "High"
]

print("\n")
print("=" * 70)

print(
    "HIGH IMPACT CUSTOMERS"
)

print("=" * 70)

print(
    f"""
Customers needing immediate attention:
{len(high_impact)}

Potential recoverable revenue:
${high_impact['recoverable_revenue'].sum():,.0f}

Average churn probability:
{round(high_impact['churn_probability'].mean(),2)}

"""
)


# TOP 10 CUSTOMERS TO SAVE


top10 = (

    df

    .nlargest(
        10,
        "priority_score"
    )

)[[
    "customer_id",
    "churn_probability",
    "risk_tier",
    "priority_score",
    "recoverable_revenue",
    "recommended_action"
]]

print("\n")
print("=" * 70)

print(
    "TOP 10 CUSTOMERS TO SAVE"
)

print("=" * 70)

print(
    top10.to_string(index=False)
)


# SAVE FINAL DATASET


df.to_csv(
    "processed/final_customers.csv",
    index=False
)


# FINAL MESSAGE


print("\n")
print("=" * 70)

print(
    "RECOMMENDATION ENGINE COMPLETE"
)

print("=" * 70)

print("""

Generated File:
final_customers.csv

Added Columns:
- recommended_action
- recommendation_reason
- expected_impact


""")



RETENTION RECOMMENDATION ANALYSIS

Top Recommended Actions:

recommended_action
Loyalty Rewards / Cashback Activation Offer     3549
Cross-Sell Credit Card or Investment Product    2842
Standard Retention Email Campaign               2556
Priority Complaint Resolution + Follow-up        627
Flexible EMI / Loan Restructuring Offer          307
Digital Re-engagement Campaign                   196
Priority Banking / Wealth Management Upgrade      23
Name: count, dtype: int64


HIGH IMPACT CUSTOMERS

Customers needing immediate attention:
4199

Potential recoverable revenue:
$2,431,347

Average churn probability:
0.45




TOP 10 CUSTOMERS TO SAVE
customer_id  churn_probability   risk_tier  priority_score  recoverable_revenue                          recommended_action
  Cust01089             0.9214    Critical        0.348475             71444.83 Loyalty Rewards / Cashback Activation Offer
  Cust05371             0.7555   High Risk        0.334460             63552.84 Loyalty Rewards / C